In [1]:
import rasterio
import geopandas as gpd
from rasterio.sample import sample_gen
import numpy as np
from pathlib import Path

In [2]:
ice_gdf = gpd.read_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_night_cut_4326_orthometric.parquet")

In [3]:
coords = [(geom.x, geom.y) for geom in ice_gdf.geometry]

In [4]:
print("ICESat CRS:", ice_gdf.crs)


ICESat CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope":

In [4]:
dem_root = Path("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped")

In [5]:
dem_paths = list(dem_root.rglob("*_egg2015.tif"))

In [6]:
dem_paths

[PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/alos/alos_dem_egg2015.tif'),
 PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/aster/aster_dem_egg2015.tif'),
 PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/copernicus/copernicus_dеm_egg2015.tif'),
 PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/fabdem/fab_dem_egg2015.tif'),
 PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/nasadem/nasa_dem_egg2015.tif'),
 PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/srtm/srtm_dem_egg2015.tif'),
 PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/tandem/tan_dem_egg2015.tif')]

In [7]:
for dem_path in dem_paths:
    name = dem_path.stem.replace("_egg2015", "")
    with rasterio.open(dem_path) as src:
        if src.crs.to_epsg() not in [4326, 4979] or ice_gdf.crs.to_epsg() not in [4326, 4979]:
            raise ValueError(f"❌ CRS не сумісні: {name} vs ICESat")

        # Витяг DEM-висот
        dem_values = list(sample_gen(src, coords))
        dem_heights = [val[0] if val and val[0] != -9999 else np.nan for val in dem_values]

        h_col = f"h_{name}"
        delta_col = f"delta_{name}"
        abs_delta_col = f"abs_delta_{name}"

        # Запис у GeoDataFrame
        ice_gdf[h_col] = dem_heights
        ice_gdf[delta_col] = ice_gdf[h_col] - ice_gdf["orthometric_height"]
        ice_gdf[abs_delta_col] = ice_gdf[delta_col].abs()

        print(f"✅ {name}: додано {h_col}, {delta_col}, {abs_delta_col}")

✅ alos_dem: додано h_alos_dem, delta_alos_dem, abs_delta_alos_dem
✅ aster_dem: додано h_aster_dem, delta_aster_dem, abs_delta_aster_dem
✅ copernicus_dеm: додано h_copernicus_dеm, delta_copernicus_dеm, abs_delta_copernicus_dеm
✅ fab_dem: додано h_fab_dem, delta_fab_dem, abs_delta_fab_dem
✅ nasa_dem: додано h_nasa_dem, delta_nasa_dem, abs_delta_nasa_dem
✅ srtm_dem: додано h_srtm_dem, delta_srtm_dem, abs_delta_srtm_dem
✅ tan_dem: додано h_tan_dem, delta_tan_dem, abs_delta_tan_dem


In [8]:
ice_gdf


,region,sc_orient,track,segment_dist,solar_elevation,segment_id,background_rate,cycle,pair,rgt,...,abs_delta_fab_dem,h_nasa_dem,delta_nasa_dem,abs_delta_nasa_dem,h_srtm_dem,delta_srtm_dem,abs_delta_srtm_dem,h_tan_dem,delta_tan_dem,abs_delta_tan_dem
2018-11-04 01:05:32.246215936,6.0,1.0,1.0,1.473420e+07,-40.593658,735621.0,1840.163382,1.0,0.0,556.0,...,3.062324,1483.172729,1.058540,1.058540,1481.0,-1.114189,1.114189,1485.023193,2.909004,2.909004
2018-11-04 01:05:32.246416128,6.0,1.0,1.0,1.473420e+07,-40.593658,735621.0,1840.163382,1.0,0.0,556.0,...,2.118110,1483.172729,0.114326,0.114326,1481.0,-2.058403,2.058403,1485.023193,1.964790,1.964790
2018-11-04 01:05:32.247016192,6.0,1.0,1.0,1.473420e+07,-40.593658,735621.0,1840.163382,1.0,0.0,556.0,...,0.851631,1483.172729,-1.152153,1.152153,1481.0,-3.324883,3.324883,1485.023193,0.698311,0.698311
2018-11-04 01:05:32.247316224,6.0,1.0,1.0,1.473420e+07,-40.593658,735621.0,1840.163382,1.0,0.0,556.0,...,2.685981,1483.172729,0.682197,0.682197,1481.0,-1.490532,1.490532,1485.023193,2.532661,2.532661
2018-11-04 01:05:32.247916288,6.0,1.0,1.0,1.473420e+07,-40.593658,735621.0,1840.163382,1.0,0.0,556.0,...,2.391548,1483.172729,0.387764,0.387764,1481.0,-1.784966,1.784966,1485.023193,2.238228,2.238228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-09-19 18:31:37.784885248,6.0,0.0,3.0,1.474884e+07,-21.542572,736349.0,4320.219828,25.0,1.0,53.0,...,11.397759,1044.153198,5.384834,5.384834,1039.0,0.231636,0.231636,1038.421875,-0.346489,0.346489
2024-09-19 18:31:37.785685248,6.0,0.0,3.0,1.474884e+07,-21.542572,736349.0,4320.219828,25.0,1.0,53.0,...,14.688286,1044.153198,2.094307,2.094307,1039.0,-3.058892,3.058892,1038.421875,-3.637017,3.637017
2024-09-19 18:31:37.787785216,6.0,0.0,3.0,1.474884e+07,-21.542572,736349.0,4320.219828,25.0,1.0,53.0,...,15.717339,1048.152588,-5.165337,5.165337,1043.0,-10.317925,10.317925,1050.464355,-2.853569,2.853569
2024-09-19 18:31:37.788485376,6.0,0.0,3.0,1.474884e+07,-21.542572,736349.0,4320.219828,25.0,1.0,53.0,...,7.544609,1048.152588,3.007393,3.007393,1043.0,-2.145195,2.145195,1050.464355,5.319160,5.319160


In [9]:
ice_gdf.to_parquet(
    "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_dems_delta_4326.parquet")
print("📦 Усі результати збережено в один файл .parquet ✅")

📦 Усі результати збережено в один файл .parquet ✅


In [10]:
gdf_utm = ice_gdf.to_crs(epsg=32635)
gdf_utm.to_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_dems_delta_32635.parquet")

In [ ]:
gdf_m = ice_gdf.to_crs(epsg=3857)
gdf_m.to_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_dems_delta_3857.parquet")